# How to Import raw actigraphy file data

Code Includes: 
- Automatically finding the "header" row from each specified actigraph file pathway
- Calculates sleep metrics including sleep fragmentation index for each participant, each day based on sleep_start and sleep_end date-times.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import os
import re

### Actigraph File pathways to Import

In [13]:
## Import file path
file_path_list = ['/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/DXA_sleep_actigraphy/DXA001_9_20_2019_2_19_00_PM_CC_24hr_Combined.csv'
] ## Can add more paths to this list as needed


### Defining Rules and Functions

In [14]:
def find_header_row(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        non_blank_index = 0
        for i, line in enumerate(f):
            if line.strip() == '':
                continue
            if (
                'Line'             in line and
                'Date'             in line and
                'Time'             in line and
                'Activity'         in line and
                'Interval Status'  in line and
                'Sleep/Wake'       in line
            ):
                return non_blank_index
            non_blank_index += 1
    raise ValueError(f'Header row not found in {filepath}')

#### Import Actigraph data files

In [16]:
## Build file list with subject_id extracted from filename
files_to_import = []

for filepath in file_path_list:
    filename = os.path.basename(filepath)

    ## extract numeric ID from filename e.g. 'DXA001_9_20...' → 'DXA_001'
    match = re.search(r'DXA(\d+)', filename, re.IGNORECASE)
    if not match:
        raise ValueError(f'Could not extract subject_id from filename: {filename}')
    subject_id = f"DXA_{int(match.group(1)):03d}"

    files_to_import.append({
        'file'      : filename,
        'subject_id': subject_id,
        'filepath'  : filepath,
        'header_row': find_header_row(filepath),
        'delimiter' : ',',
        'usecols'   : list(range(12))
    })

for f in files_to_import:
    print(f['subject_id'], '→', f['file'], '→ header row:', f['header_row'])

DXA_001 → DXA001_9_20_2019_2_19_00_PM_CC_24hr_Combined.csv → header row: 246


In [17]:
# ── Load all files into one combined dataframe ────────────────────────────────
dfs = []

for f in files_to_import:
    df = pd.read_csv(
        f['filepath'],
        header    = f['header_row'],
        delimiter = f['delimiter'],
        usecols   = f['usecols']
    )
    df.columns = df.columns.str.strip().str.lower().str.replace('"', '')

    # assign subject_id and study from filename — no dependency on file contents
    df['subject_id'] = f['subject_id']
    df['study']      = 'DXA'

    dfs.append(df)

actigraphy_df = pd.concat(dfs, ignore_index=True)

print('===================')
print(f"actigraphy_df shape : {actigraphy_df.shape}")
print('===================')
print(f"subject_id values   : {actigraphy_df['subject_id'].unique()}")
print('===================')

actigraphy_df shape : (25994, 14)
subject_id values   : ['DXA_001']


In [18]:
actigraphy_df.head()

,line,date,time,off-wrist status,activity,marker,white light,red light,green light,blue light,sleep/wake,interval status,subject_id,study
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DXA_001,DXA
1,1.0,9/20/19,2:19:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA
2,2.0,9/20/19,2:20:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA
3,3.0,9/20/19,2:21:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA
4,4.0,9/20/19,2:22:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA


### Summary Actigraphy DF (scored)

In [19]:
actigraphy_df['date_time24'] = pd.to_datetime(actigraphy_df['date'] + ' ' + actigraphy_df['time'])
actigraphy_df.columns = actigraphy_df.columns.str.replace(' ', '_')

actigraphy_df.head()

/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_77573/2421419887.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  actigraphy_df['date_time24'] = pd.to_datetime(actigraphy_df['date'] + ' ' + actigraphy_df['time'])


,line,date,time,off-wrist_status,activity,marker,white_light,red_light,green_light,blue_light,sleep/wake,interval_status,subject_id,study,date_time24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DXA_001,DXA,NaT
1,1.0,9/20/19,2:19:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA,2019-09-20 14:19:00
2,2.0,9/20/19,2:20:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA,2019-09-20 14:20:00
3,3.0,9/20/19,2:21:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA,2019-09-20 14:21:00
4,4.0,9/20/19,2:22:00 PM,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,ACTIVE,DXA_001,DXA,2019-09-20 14:22:00


In [20]:
scored_actigraphy_df = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Data_analysis/sleep_merged_df_cleaned_062326.xlsx', index_col=0)

scored_actigraphy_df.columns = scored_actigraphy_df.columns.str.lower().str.replace(' ', '_')

dt_columns = ['start_date', 'start_time', 'end_date',
       'end_time', 'start_datetime', 'end_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h']

for col in dt_columns:
    scored_actigraphy_df[col] = pd.to_datetime(scored_actigraphy_df[col])

scored_actigraphy_df.reset_index(drop=True, inplace=True)
scored_actigraphy_df.head()

/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_77573/3430119275.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_actigraphy_df[col] = pd.to_datetime(scored_actigraphy_df[col])
/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_77573/3430119275.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_actigraphy_df[col] = pd.to_datetime(scored_actigraphy_df[col])


,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-07-07 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
1,DXA_001,DXA001,Sleep,2,Saturday,NaT,NaN,NaT,NaT,NaN,...,NaN,EXCLUDED - >15% of the day off-wrist,DXA,NaN,NaT,NaT,NaN,NaN,NaT,NaT
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-07-07 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-07-07 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-07-07 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00


# Fragmentation Index Troubleshooting

In [21]:
# ### For calculating fragmentation index
# from itertools import groupby

# def count_immobile_bouts(activity_series):
#     """
#     Count number of continuous immobile bouts (consecutive epochs where activity < 40).
#     Each unbroken run of immobile epochs = 1 bout.
#     """
#     immobile_flags = (activity_series < 40).tolist()
#     bouts = [list(group) for key, group in groupby(immobile_flags) if key]
#     return len(bouts)

In [22]:
## Fragmentation Index Calculation

## A = mobile epochs (activity count >= 40)
## B = sandwiched immobile epochs (activity < 40, but both pre- proceeding epochs >=40 ac)
## C = total immobile epochs (ac <40)

activity_threshold = 40

def compute_fragmentation_index(activity_series):
    """
    Compute Fragmentation index from series of activity counts > the sleep period (sleep start to sleep end)
    returns fragmentation index as a percentage, plus A, B, C, components
    fragmentation index = ((A+B)/C)*100
    """

    act = activity_series.reset_index(drop=True)

    mobile_mask = act >= activity_threshold
    A = int(mobile_mask.sum())

    immobile_mask = act < activity_threshold
    C = int(immobile_mask.sum())

    B = 0
    for i in range(1,len(act)-1):
        if (
            act[i] < activity_threshold and
            act[i-1] >= activity_threshold and
            act[i+1] >= activity_threshold
        ):
            B +=1

    fragmentation_index = round(((A+B)/C)*100) if C >0 else None

    return {
        'A_mobile_epochs': A,
        'B_sandwiched_immobile': B,
        'C_total_immobile' : C,
        'fragmentation_index': fragmentation_index,
    }

## Calculate Sleep Metrics

In [23]:
## keep only nights where flag_actigraph is null/empty (not flagged)
valid_scored_df = scored_actigraphy_df[scored_actigraphy_df['flag_actigraph'].isna()].copy()

print(f"Total nights before flag filter : {len(scored_actigraphy_df)}")
print(f"Total nights after flag filter  : {len(valid_scored_df)}")
print(f"Nights dropped                  : {len(scored_actigraphy_df) - len(valid_scored_df)}")

Total nights before flag filter : 3062
Total nights after flag filter  : 2585
Nights dropped                  : 477


In [24]:
## Standardize datetimes and confirm they're in datetime format

valid_scored_df['start_datetime'] = pd.to_datetime(valid_scored_df['start_datetime'])
valid_scored_df['sleep_mid_dt']   = pd.to_datetime(valid_scored_df['sleep_mid_dt'])
valid_scored_df['sleep_onset_plus4h'] = pd.to_datetime(valid_scored_df['sleep_onset_plus4h'])
valid_scored_df['end_datetime'] = pd.to_datetime(valid_scored_df['end_datetime'])
## actigraphy_df pd.to_datime
actigraphy_df['date_time24']      = pd.to_datetime(actigraphy_df['date_time24'])

## doublecheck
print("Dtypes confirmed:")
print(valid_scored_df[['start_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h', 'end_datetime']].dtypes)
print(actigraphy_df['date_time24'].dtype)

Dtypes confirmed:
start_datetime        datetime64[ns]
sleep_mid_dt          datetime64[ns]
sleep_onset_plus4h    datetime64[ns]
end_datetime          datetime64[ns]
dtype: object
datetime64[ns]


In [25]:
valid_scored_df.head()

,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-07-07 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-07-07 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-07-07 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-07-07 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00
5,DXA_001,DXA001,Sleep,6,Wednesday,2019-09-25,Wed,2026-07-07 23:31:00,2019-09-26,Thu,...,27.33,NaN,DXA,NaN,2019-09-25 23:31:00,2019-09-26 06:59:00,0.311111,448.0,2019-09-26 03:15:00,2019-09-26 03:31:00


In [26]:
## Fragmentation Index Calculation
## A = mobile epochs (activity count >= 40)
## B = sandwiched immobile epochs (activity < 40, but both pre- proceeding epochs >=40 ac)
## C = total immobile epochs (ac <40)
activity_threshold = 40

def compute_fragmentation_index(activity_series):
    """
    Compute Fragmentation index from series of activity counts > the sleep period (sleep start to sleep end)
    returns fragmentation index as a percentage, plus A, B, C, components
    fragmentation index = ((A+B)/C)*100
    """
    act = activity_series.reset_index(drop=True)

    mobile_mask = act >= activity_threshold
    A = int(mobile_mask.sum())

    immobile_mask = act < activity_threshold
    C = int(immobile_mask.sum())

    B = 0
    for i in range(1,len(act)-1):
        if (
            act[i] < activity_threshold and
            act[i-1] >= activity_threshold and
            act[i+1] >= activity_threshold
        ):
            B +=1
            
    fragmentation_index = round(((A+B)/C)*100, 2) if C >0 else None

    return {
        'A_mobile_epochs': A,
        'B_sandwiched_immobile': B,
        'C_total_immobile' : C,
        'fragmentation_index': fragmentation_index,
    }

In [27]:
# ── Full sleep metrics loop (cleaned) ────────────────────────────────────────

full_sleep_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['end_datetime']      

    # slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    # --- sleep/wake classification ---
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    full_sleep_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

full_sleep_results_df = pd.DataFrame(full_sleep_records)

print(f"Results shape: {full_sleep_results_df.shape}")
# print(full_sleep_results_df.to_string(index=False))

full_sleep_results_df.head(15)

Results shape: (2585, 13)


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24.0,3.0,491.0,5.50
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36.0,0.0,612.0,5.88
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43.0,5.0,525.0,9.14
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13.0,0.0,458.0,2.84
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28.0,1.0,420.0,6.90
5,DXA_001,10,2019-09-30 00:16:00,2019-09-30 08:06:00,470,438,32,32,93.19,25.0,1.0,445.0,5.84
6,DXA_001,11,2019-09-30 23:37:00,2019-10-01 07:18:00,461,429,32,32,93.06,28.0,0.0,433.0,6.47
7,DXA_001,12,2019-10-02 00:32:00,2019-10-02 07:59:00,447,414,33,33,92.62,22.0,0.0,425.0,5.18
8,DXA_001,13,2019-10-03 00:34:00,2019-10-03 06:29:00,355,333,22,22,93.80,19.0,1.0,336.0,5.95
9,DXA_001,14,2019-10-04 00:11:00,2019-10-04 07:21:00,430,411,19,19,95.58,18.0,0.0,412.0,4.37


### Stop

Hurray!!!